In [ ]:
!pip install medmnist
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import medmnist
from medmnist import INFO
from torch.utils.data import DataLoader
import pennylane as qml
import time

# Setup and Full Dataset Loading
print("Setting up RetinaMNIST Full Dataset...")
data_flag = 'retinamnist'
info = INFO[data_flag]
n_classes = len(info['label'])
DataClass = getattr(medmnist, info['python_class'])

# Define data transformations
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# Loading the dataset
train_dataset = DataClass(split='train', transform=data_transform, download=True)
val_dataset = DataClass(split='val', transform=data_transform, download=True)

batch_size = 16
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)

print(f"Full Dataset Loaded! Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")


# Quantum Circuit
n_qubits = 6
q_layers = 2

# setting up my VQC:-
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def quantum_net(inputs, weights):
    # Amplitude Embedding to encode 64 features into 6 qubits
    qml.AmplitudeEmbedding(features=inputs, wires=range(n_qubits), normalize=True)
    
    # Highly expressive entanglement layer
    qml.StronglyEntanglingLayers(weights=weights, wires=range(n_qubits))
    
    # Measuring Pauli-Z expectation value for all 6 qubits
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


# Hybrid QNN Architecture 
class HybridAmplitudeQNN(nn.Module):
    def __init__(self, num_classes=5):
        super(HybridAmplitudeQNN, self).__init__()
        
        # Classical CNN Feature Extractor (RGB 3-channel input)
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        self.flattened_size = 32 * 7 * 7
        
        # Bottleneck mapping to exactly 64 continuous features.
        self.fc_reduce = nn.Linear(self.flattened_size, 64)
        
        # Quantum Layer Setup
        weight_shapes = {"weights": (q_layers, n_qubits, 3)}
        self.qlayer = qml.qnn.TorchLayer(quantum_net, weight_shapes)
        
        # (Mapping 6 qubit measurements to 5 classes)
        self.fc_out = nn.Linear(n_qubits, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) 
        x = F.relu(self.fc_reduce(x))
        
        # CRITICAL PHYSICS HACK: L2 Normalization to satisfy Tr(ρ)=1
        x = F.normalize(x, p=2, dim=1, eps=1e-12)
        
        # Quantum Processing
        x = self.qlayer(x)
        
        # Classical Output Mapping
        x = self.fc_out(x)
        return x

# Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = HybridAmplitudeQNN(num_classes=n_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Model training
epochs = 5
print("\n--- Starting Hybrid Amplitude QNN Training (Full Dataset) ---")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    start_time = time.time()
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device, dtype=torch.float32), labels.squeeze().to(device, dtype=torch.long)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # PREVENTING GRADIENT EXPLOSIONS (Troubleshooting implemented here)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_acc = 100 * correct / total
    epoch_time = time.time() - start_time
    
    print(f"Epoch [{epoch+1}/{epochs}] | Time: {epoch_time:.2f}s | "
          f"Loss: {running_loss/len(train_loader):.4f} | Training Acc: {epoch_acc:.2f}%")

# Save the final Hybrid model
torch.save(model.state_dict(), "hybrid_retinamnist_amplitude_full.pth")
print("\nFull Dataset Hybrid model saved as 'hybrid_retinamnist_amplitude_full.pth'")
